In [61]:
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

## Load database

In [62]:
df_patients = pd.read_csv("../database/patients.csv")
df_services_weekly = pd.read_csv("../database/services_weekly.csv")
df_staff_schedule = pd.read_csv("../database/staff_schedule.csv")
df_staff = pd.read_csv("../database/staff.csv")

print(f"Shape of df_patients: {df_patients.shape}")
print(f"Shape of df_services_weekly: {df_services_weekly.shape}")
print(f"Shape of df_staff_schedule: {df_staff_schedule.shape}")
print(f"Shape of df_staff: {df_staff.shape}")

Shape of df_patients: (1000, 7)
Shape of df_services_weekly: (208, 10)
Shape of df_staff_schedule: (6552, 6)
Shape of df_staff: (110, 4)


In [63]:
print(df_patients.head(2))
print(df_services_weekly.head(2))
print(df_staff_schedule.head(2))
print(df_staff.head(2))

     patient_id               name  age arrival_date departure_date  service  \
0  PAT-09484753  Richard Rodriguez   24   2025-03-16     2025-03-22  surgery   
1  PAT-f0644084     Shannon Walker    6   2025-12-13     2025-12-14  surgery   

   satisfaction  
0            61  
1            83  
   week  month    service  available_beds  patients_request  \
0     1      1  emergency              32                76   
1     1      1    surgery              45               130   

   patients_admitted  patients_refused  patient_satisfaction  staff_morale  \
0                 32                44                    67            70   
1                 45                85                    83            78   

  event  
0  none  
1   flu  
   week      staff_id    staff_name    role    service  present
0     1  STF-b77cdc60  Allison Hill  doctor  emergency        1
1     2  STF-b77cdc60  Allison Hill  doctor  emergency        1
       staff_id    staff_name    role    service
0  STF-5c

## Merge database

In [64]:
df_patients['arrival_date'] = pd.to_datetime(df_patients['arrival_date'])
df_patients['departure_date'] = pd.to_datetime(df_patients['departure_date'])
df_patients['month'] = df_patients['arrival_date'].dt.month
df_patients['week'] = (df_patients['arrival_date'].dt.day - 1) // 7 + 1

# Aggregate staff schedule
df_staff_weekly = (
    df_staff_schedule
    .groupby(["week", "service"])
    .agg(
        total_staff=('staff_id', 'nunique'),
        staff_present=("present", "sum"),
        staff_absens=("present", lambda x: (x == 0).sum())
    )
    .reset_index()
)

# Add staff information
df_staff_weekly_detail = df_staff_schedule.merge(
    df_staff,
    on="staff_id",
    how="left",
    suffixes=("_schedule", "_master")
)

# Combine weekly service + staff information
df_service = df_services_weekly.merge(
    df_staff_weekly,
    on=["week", "service"],
    how="left"
)

df = df_patients.merge(
    df_service,
    on=["month", "week", "service"],
    how="left"
)

print("Final shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

Final shape: (1000, 19)

Columns:
['patient_id', 'name', 'age', 'arrival_date', 'departure_date', 'service', 'satisfaction', 'month', 'week', 'available_beds', 'patients_request', 'patients_admitted', 'patients_refused', 'patient_satisfaction', 'staff_morale', 'event', 'total_staff', 'staff_present', 'staff_absens']


## Fill missing value

In [65]:
# Create service statistics
service_stats = (
    df_services_weekly
    .groupby("service")
    .agg(
        available_beds=("available_beds", "mean"),
        patients_request=("patients_request", "mean"),
        patients_admitted=("patients_admitted", "mean"),
        patients_refused=("patients_refused", "mean"),
        patient_satisfaction=("patient_satisfaction", "mean"),
        staff_morale=("staff_morale", "mean")
    )
    .reset_index()
)

In [66]:
df = df.merge(
    service_stats,
    on="service",
    how="left",
    suffixes=("", "_service_avg")
)

# Fill missing values using service average
fill_mapping = {
    "available_beds": "available_beds_service_avg",
    "patients_request": "patients_request_service_avg",
    "patients_admitted": "patients_admitted_service_avg",
    "patients_refused": "patients_refused_service_avg",
    "patient_satisfaction": "patient_satisfaction_service_avg",
    "staff_morale": "staff_morale_service_avg"
}

for target_col, source_col in fill_mapping.items():
    df[target_col] = df[target_col].fillna(df[source_col])

df = df.drop(columns=list(fill_mapping.values()))

In [67]:
staff_stats = (
    df_staff_schedule
    .groupby("service")
    .agg(
        total_staff=("staff_id", "nunique"),
        staff_present=("present", "sum"),
        staff_absens=("present", lambda x: (x == 0).sum())
    )
    .reset_index()
)

df = df.merge(
    staff_stats,
    on="service",
    how="left",
    suffixes=("", "_service_avg")
)

In [68]:
# Fill missing values for staff statistics using service average
df["total_staff"] = df["total_staff"].fillna(df["total_staff_service_avg"])
df["staff_present"] = df["staff_present"].fillna(df["staff_present_service_avg"])
df["staff_absens"] = df["staff_absens"].fillna(df["staff_absens_service_avg"])

df = df.drop(
    columns=[
        "total_staff_service_avg",
        "staff_present_service_avg",
        "staff_absens_service_avg"
    ]
)

In [69]:
# Round the values  to 2 decimal places
numeric_cols = [
    "available_beds",
    "patients_request",
    "patients_admitted",
    "patients_refused",
    "patient_satisfaction",
    "staff_morale",
    "total_staff",
    "staff_present",
    "staff_absens"
]

# Make available beds column into integer type
df["available_beds"] = df["available_beds"].astype(int)
df['total_staff'] = df['total_staff'].astype(int)

df[numeric_cols] = df[numeric_cols].round(2)

# Drop the "event" column if it exists
df = df.drop(columns=["event"])

In [70]:
df.head(2)

,patient_id,name,age,arrival_date,departure_date,service,satisfaction,month,week,available_beds,patients_request,patients_admitted,patients_refused,patient_satisfaction,staff_morale,total_staff,staff_present,staff_absens
0,PAT-09484753,Richard Rodriguez,24,2025-03-16,2025-03-22,surgery,61,3,3,37,43.1,32.42,10.67,79.27,72.63,25,783.0,517.0
1,PAT-f0644084,Shannon Walker,6,2025-12-13,2025-12-14,surgery,83,12,2,37,43.1,32.42,10.67,79.27,72.63,25,783.0,517.0


In [71]:
df.value_counts()

patient_id    name               age  arrival_date  departure_date  service           satisfaction  month  week  available_beds  patients_request  patients_admitted  patients_refused  patient_satisfaction  staff_morale  total_staff  staff_present  staff_absens
PAT-003ce690  Larry Dixon        29   2025-01-19    2025-01-21      ICU               60            1      3     20              21.00             20.00              1.00              82.00                 89.00         34           0.0            34.0            1
PAT-a5027bf8  Rodney Morales     38   2025-03-23    2025-03-27      general_medicine  61            3      4     46              82.12             44.85              37.27             81.23                 73.10         28           859.0          597.0           1
PAT-a1bec809  Catherine Frazier  46   2025-04-11    2025-04-25      ICU               82            4      2     14              15.17             12.46              2.71              81.62                 7

In [72]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 18 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   patient_id            1000 non-null   object        
 1   name                  1000 non-null   object        
 2   age                   1000 non-null   int64         
 3   arrival_date          1000 non-null   datetime64[ns]
 4   departure_date        1000 non-null   datetime64[ns]
 5   service               1000 non-null   object        
 6   satisfaction          1000 non-null   int64         
 7   month                 1000 non-null   int32         
 8   week                  1000 non-null   int32         
 9   available_beds        1000 non-null   int64         
 10  patients_request      1000 non-null   float64       
 11  patients_admitted     1000 non-null   float64       
 12  patients_refused      1000 non-null   float64       
 13  patient_satisfactio

## Develop data into 10,000 rows

In [73]:
from faker import Faker
import numpy as np

# ============================================================
# 1. CONFIGURATION
# ============================================================

np.random.seed(42)
Faker.seed(42)

fake = Faker()

TARGET_ROWS = 10_000
CURRENT_ROWS = len(df)
ADDITIONAL_ROWS = TARGET_ROWS - CURRENT_ROWS

print("Current rows :", CURRENT_ROWS)
print("Rows to add  :", ADDITIONAL_ROWS)


# ============================================================
# 2. REFERENCE DISTRIBUTIONS
# ============================================================

# ------------------------------------------------------------
# Service distribution
# ------------------------------------------------------------

service_distribution = (
    df["service"]
    .value_counts(normalize=True)
)

services = service_distribution.index.tolist()
service_probabilities = service_distribution.values


# ------------------------------------------------------------
# Age distribution
# ------------------------------------------------------------

age_mean = df["age"].mean()
age_std = df["age"].std()


# ------------------------------------------------------------
# Patient satisfaction distribution
# ------------------------------------------------------------

satisfaction_mean = df["satisfaction"].mean()
satisfaction_std = df["satisfaction"].std()


# ------------------------------------------------------------
# Hospital statistics by service
# ------------------------------------------------------------

service_stats = (
    df
    .groupby("service")
    .agg(
        available_beds_mean=("available_beds", "mean"),
        available_beds_std=("available_beds", "std"),

        patients_request_mean=("patients_request", "mean"),
        patients_request_std=("patients_request", "std"),

        patients_admitted_mean=("patients_admitted", "mean"),
        patients_admitted_std=("patients_admitted", "std"),

        patients_refused_mean=("patients_refused", "mean"),
        patients_refused_std=("patients_refused", "std"),

        patient_satisfaction_mean=("patient_satisfaction", "mean"),
        patient_satisfaction_std=("patient_satisfaction", "std"),

        staff_morale_mean=("staff_morale", "mean"),
        staff_morale_std=("staff_morale", "std"),

        total_staff_mean=("total_staff", "mean"),
        total_staff_std=("total_staff", "std"),

        staff_present_mean=("staff_present", "mean"),
        staff_present_std=("staff_present", "std"),

        staff_absens_mean=("staff_absens", "mean"),
        staff_absens_std=("staff_absens", "std")
    )
)

# Prevent NaN standard deviations
service_stats = service_stats.fillna(0)


# ============================================================
# 3. GENERATE SYNTHETIC DATA
# ============================================================

synthetic_patients = []


for _ in range(ADDITIONAL_ROWS):

    # --------------------------------------------------------
    # Patient ID
    # --------------------------------------------------------

    patient_id = f"PAT-{fake.hexify(text='^^^^^^^^')}"


    # --------------------------------------------------------
    # Name
    # --------------------------------------------------------

    name = fake.name()


    # --------------------------------------------------------
    # Age
    # --------------------------------------------------------

    age = int(
        np.clip(
            np.random.normal(
                age_mean,
                age_std
            ),
            0,
            100
        )
    )


    # --------------------------------------------------------
    # Service
    # --------------------------------------------------------

    service = np.random.choice(
        services,
        p=service_probabilities
    )


    # --------------------------------------------------------
    # Arrival date
    # --------------------------------------------------------

    start_date = pd.Timestamp("2025-01-01")
    end_date = pd.Timestamp("2025-12-31")

    days_range = (
        end_date - start_date
    ).days

    random_days = np.random.randint(
        0,
        days_range + 1
    )

    arrival_date = (
        start_date +
        pd.Timedelta(days=int(random_days))
    )


    # --------------------------------------------------------
    # Departure date
    # --------------------------------------------------------

    length_of_stay = np.random.randint(
        1,
        15
    )

    departure_date = (
        arrival_date +
        pd.Timedelta(days=int(length_of_stay))
    )


    # --------------------------------------------------------
    # Calendar features
    # --------------------------------------------------------

    month = arrival_date.month

    week = (
        (arrival_date.day - 1) // 7
    ) + 1


    # --------------------------------------------------------
    # Patient satisfaction
    # --------------------------------------------------------

    satisfaction = int(
        np.clip(
            np.random.normal(
                satisfaction_mean,
                satisfaction_std
            ),
            0,
            100
        )
    )


    # ========================================================
    # HOSPITAL FEATURES
    # ========================================================

    stats = service_stats.loc[service]


    # --------------------------------------------------------
    # Available beds
    # --------------------------------------------------------

    available_beds = int(
        max(
            0,
            round(
                np.random.normal(
                    stats["available_beds_mean"],
                    stats["available_beds_std"]
                )
            )
        )
    )


    # --------------------------------------------------------
    # Patient requests
    # --------------------------------------------------------

    patients_request = max(
        0,
        np.random.normal(
            stats["patients_request_mean"],
            stats["patients_request_std"]
        )
    )


    # --------------------------------------------------------
    # Patients admitted
    # --------------------------------------------------------

    patients_admitted = max(
        0,
        np.random.normal(
            stats["patients_admitted_mean"],
            stats["patients_admitted_std"]
        )
    )


    # --------------------------------------------------------
    # Patients refused
    # --------------------------------------------------------

    patients_refused = max(
        0,
        np.random.normal(
            stats["patients_refused_mean"],
            stats["patients_refused_std"]
        )
    )


    # --------------------------------------------------------
    # Patient satisfaction
    # --------------------------------------------------------

    patient_satisfaction = np.clip(
        np.random.normal(
            stats["patient_satisfaction_mean"],
            stats["patient_satisfaction_std"]
        ),
        0,
        100
    )


    # --------------------------------------------------------
    # Staff morale
    # --------------------------------------------------------

    staff_morale = np.clip(
        np.random.normal(
            stats["staff_morale_mean"],
            stats["staff_morale_std"]
        ),
        0,
        100
    )


    # --------------------------------------------------------
    # Total staff
    # --------------------------------------------------------

    total_staff = int(
        max(
            1,
            round(
                np.random.normal(
                    stats["total_staff_mean"],
                    stats["total_staff_std"]
                )
            )
        )
    )


    # --------------------------------------------------------
    # Staff present
    # --------------------------------------------------------

    staff_present = max(
        0,
        np.random.normal(
            stats["staff_present_mean"],
            stats["staff_present_std"]
        )
    )


    # --------------------------------------------------------
    # Staff absent
    # --------------------------------------------------------

    staff_absens = max(
        0,
        np.random.normal(
            stats["staff_absens_mean"],
            stats["staff_absens_std"]
        )
    )


    # --------------------------------------------------------
    # Store row
    # --------------------------------------------------------

    synthetic_patients.append({

        "patient_id": patient_id,

        "name": name,

        "age": age,

        "arrival_date": arrival_date,

        "departure_date": departure_date,

        "service": service,

        "satisfaction": satisfaction,

        "month": month,

        "week": week,

        "available_beds": available_beds,

        "patients_request": patients_request,

        "patients_admitted": patients_admitted,

        "patients_refused": patients_refused,

        "patient_satisfaction": patient_satisfaction,

        "staff_morale": staff_morale,

        "total_staff": total_staff,

        "staff_present": staff_present,

        "staff_absens": staff_absens
    })


# ============================================================
# 4. CREATE SYNTHETIC DATAFRAME
# ============================================================

df_synthetic = pd.DataFrame(
    synthetic_patients
)


print("\nSynthetic shape:")
print(df_synthetic.shape)


# ============================================================
# 5. COMBINE ORIGINAL + SYNTHETIC
# ============================================================

df_10k = pd.concat(
    [
        df,
        df_synthetic
    ],
    ignore_index=True
)


# ============================================================
# 6. ENSURE COLUMN ORDER
# ============================================================

columns = [
    "patient_id",
    "name",
    "age",
    "arrival_date",
    "departure_date",
    "service",
    "satisfaction",
    "month",
    "week",
    "available_beds",
    "patients_request",
    "patients_admitted",
    "patients_refused",
    "patient_satisfaction",
    "staff_morale",
    "total_staff",
    "staff_present",
    "staff_absens"
]

df_10k = df_10k[columns]


# ============================================================
# 7. ROUND NUMERIC VALUES
# ============================================================

decimal_columns = [
    "patients_request",
    "patients_admitted",
    "patients_refused",
    "patient_satisfaction",
    "staff_morale",
    "staff_present",
    "staff_absens"
]

df_10k[decimal_columns] = (
    df_10k[decimal_columns]
    .round(2)
)

Current rows : 1000
Rows to add  : 9000

Synthetic shape:
(9000, 18)


In [75]:
df_10k

,patient_id,name,age,arrival_date,departure_date,service,satisfaction,month,week,available_beds,patients_request,patients_admitted,patients_refused,patient_satisfaction,staff_morale,total_staff,staff_present,staff_absens
0,PAT-09484753,Richard Rodriguez,24,2025-03-16,2025-03-22,surgery,61,3,3,37,43.10,32.42,10.67,79.27,72.63,25,783.00,517.00
1,PAT-f0644084,Shannon Walker,6,2025-12-13,2025-12-14,surgery,83,12,2,37,43.10,32.42,10.67,79.27,72.63,25,783.00,517.00
2,PAT-ac6162e4,Julia Torres,24,2025-06-29,2025-07-05,general_medicine,83,6,5,46,82.12,44.85,37.27,81.23,73.10,28,859.00,597.00
3,PAT-3dda2bb5,Crystal Johnson,32,2025-10-12,2025-10-23,emergency,81,10,2,22,119.10,22.79,96.31,77.88,73.56,39,1225.00,803.00
4,PAT-08591375,Garrett Lin,25,2025-02-18,2025-02-25,ICU,76,2,3,14,15.17,12.46,2.71,81.62,70.98,34,1063.00,705.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,PAT-64e96b06,Travis Wiggins,39,2025-10-09,2025-10-18,surgery,59,10,2,38,27.53,39.41,12.52,83.83,80.24,25,811.88,490.95
9996,PAT-7070191a,Brandi Poole,62,2025-06-12,2025-06-13,general_medicine,80,6,2,47,91.86,44.55,77.79,78.86,76.57,28,906.20,444.74
9997,PAT-889aa621,Louis Charles,16,2025-05-05,2025-05-06,surgery,68,5,1,35,37.39,34.46,11.46,76.64,77.41,25,914.55,400.68
9998,PAT-2b7989b0,Julie Alvarez,44,2025-12-19,2025-12-21,emergency,90,12,3,22,124.46,24.58,103.98,76.24,72.13,39,1178.85,778.45
